# Introduction
This notebook will perform extensive data analysis regarding bird bioacoustic monitoring. The data will be taken from the article "Automated identification of individual birds by song enables multi-year recapture from passive acoustic monitoring data", and it is located under resources/localized_events.csv. 

## Package Imports

In [ ]:
# First, we must import the necessary packages and libraries
import pandas as pd
import ast
import math
from pathlib import Path 
from plotnine import ( 
    ggplot, 
    aes, 
    geom_point,
    geom_bar,
    facet_wrap,
    labs,
    scale_fill_manual,
    theme_minimal
)

## Data Loading

Then, we will need to load in the data as a dataframe or just fetch the binary if it exists

In [ ]:
path = Path("resources/localized_events.parquet")

if path.exists():
    print("Exists, so just load in binary file")
    all_loaded_events_df = pd.read_parquet("resources/localized_events.parquet")
else:
    print("Does not exist, so we create it")
    all_loaded_events_df = pd.read_csv("resources/localized_events.csv")
    all_loaded_events_df.to_parquet("resources/localized_events.parquet")


Now, we will take a subset of the column variables in the data spreadsheet.

In [ ]:
subst_loaded_events_df = all_loaded_events_df[
    [
    # Metadata
    'array',
    'data_split',
    'relative_position',
    'mic_distances_m',
    'start_timestamp',
    'duration',

    # Automated models
    'ovenbird_cluster_labels',
    'baseline_resnet18_cluster_labels',
    'hawkears_cluster_labels',
    'birdnet_cluster_labels',
    ]
]

## Exploratory Data Analysis
We will begin by exploring the distributions of the data to identify possible outliers, trends, and occurrences. 

In [ ]:
# add a new column individual so it does not conflict with python
subst_loaded_events_df['individual'] = all_loaded_events_df['class']

# Convert to datetime objects
subst_loaded_events_df['start_timestamp'] = pd.to_datetime(subst_loaded_events_df['start_timestamp'], utc=True)
subst_loaded_events_df['duration'] = pd.to_timedelta(subst_loaded_events_df['duration'], unit='minute')

In [ ]:
# Create a new column just for the end_timestamp -> datetime object
subst_loaded_events_df['end_timestamp'] = (subst_loaded_events_df['start_timestamp'] + subst_loaded_events_df['duration'])

In [ ]:
# Set an event key for all the rows
subst_loaded_events_df['event_key'] = range(len(subst_loaded_events_df))

In [ ]:
# .value_counts() creates a lookup table, (individual: counts) for all individuals
# .map() takes that lookup table and assigns it to those individuals using 'individual' as the key
# This new column 'count' is assigned with that counts for each individual
lookup_individual_counts = subst_loaded_events_df['individual'].value_counts()
subst_loaded_events_df['count'] = subst_loaded_events_df['individual'].map(
    lookup_individual_counts
)

In [ ]:
# Save to under resources
subst_loaded_events_df.to_parquet('resources/subset_loaded_events.parquet')
subst_loaded_events_df.to_csv('resources/subset_loaded_events.csv')

In [ ]:
# Creates a ggplot object 
(
    ggplot(subst_loaded_events_df, aes(x='factor(individual)', fill='count > 30'))
    + geom_bar()
    + scale_fill_manual(
        values={True: 'blue', False: 'red'}
    )
    + labs(
        title='Song Count per Individual',
        x='Individual',
        y='Count'
    )
)

Now, we want to plot the recordings according to the relative (x, y) coordinates, they will be labeled according to the individual they belong to & faceted by the sites they belong to.

In [ ]:
# Create the figure for this, using ast package to unpack the str
fig_df = pd.DataFrame()
fig_df['x'] = subst_loaded_events_df['relative_position'].apply(
    lambda p: ast.literal_eval(p)[0]
)
fig_df['y'] = subst_loaded_events_df['relative_position'].apply(
    lambda p: ast.literal_eval(p)[1]
)
fig_df['individual'] = subst_loaded_events_df['individual'].astype("category")

In [ ]:
(
    ggplot(fig_df, aes('x', 'y', fill = 'individual')) +
        geom_point() 
)

Now, we will facet grid them by the site.

In [ ]:
# Introduce the site column to enable facet gridding
fig_df['site'] = subst_loaded_events_df['array']

In [ ]:
(
    ggplot(fig_df, aes('x', 'y', fill = 'individual')) +
        geom_point() +
        facet_wrap('site') +
        labs(
        title='Ovenbird Song Positions by Array',
        x='Easting (m)',
        y='Northing (m)',
        color='Individual'
        ) +
        theme_minimal()
)

## Acoustic Event Processing
The overlaps of all the bird recordings will be found, and that information will be used to determine bird interactions.
- If two bird recordings overlap, that may or may not be an interaction (further analysis is required)
- If two bird recordings do not overlap, that is not an interaction

In [ ]:
from pandas import DataFrame

### Data Preparation

In [ ]:
def create_event_tuple_list_from_df(metrics_df: DataFrame) -> list[tuple]:
    '''
    Return a list of event tuples (key, id, start, end, x, y) from metrics_df
    '''
    return list(metrics_df[['individual', 'scale_time_start', 'scale_time_end', 'position_x', 'position_y']].itertuples(index=True, name=None))

### Temporal Overlap Detection

In [ ]:
def time_overlap(event_1, event_2) -> bool:
    '''
    Given two events: Event 1 and Event 2 (key, id, start, end, x, y), True if they overlap, false if not \n
    Invariant holds if the two events are presented in ascending sorted order \n
    O(1) time complexity, simply returns true or false
    '''
    #sorted_events = sorted([event_1, event_2], key=lambda e: e[2])

    _, _, _, event_1_end, _, _ = event_1
    _, _, event_2_start, _, _, _ = event_2

    return event_2_start < event_1_end

In [ ]:
# Find the overlap from all the events
def time_overlaps(events) -> list[tuple]:
    '''
    Finds a list of all event overlaps, then returns a list of them as tuples
    '''
    # We need to find all the ways the events overlap, then return a tuple of event ids 
    events_time_overlapped: list[tuple] = []
    events = sorted(events, key=lambda x: x[2]) # key must use a function for comparison

    for i, event_1 in enumerate(events): 
        #ind_overlaps = overlap_individual(event=event, i=i+1, total_events=events, sort=True)
        #events_overlapped.extend(ind_overlaps)

        for event_2 in events[i+1:]: # compare to the rest after it
            if time_overlap(event_1, event_2): 
                events_time_overlapped.append((event_1[0], event_2[0])) 
            else:
                break
        
    return events_time_overlapped

Naively it works, but the overlap count appears to be very low. This suggests that the individual birds in the sites typically do not interrupt one another.

Now, we should specify upper and lower bounds to increase the overlaps. Eg. Two events overlap if they occurred within x minutes of one another.

In [ ]:
cols = subst_loaded_events_df.columns
metrics_df = subst_loaded_events_df[['event_key', 'individual', 'start_timestamp', 'end_timestamp', 'relative_position']]

metrics_df['relative_position'] = metrics_df['relative_position'].apply(
                                    ast.literal_eval
                                  )

metrics_df['position_x'] = metrics_df['relative_position'].apply(
    lambda p: p[0]
)

metrics_df['position_y'] = metrics_df['relative_position'].apply(
    lambda p: p[1]
)

In [ ]:
# event lower and upper bounds
lower_s = 5
upper_e = 5

In [ ]:
# create a new column that scales the start and end
metrics_df['scale_time_start'] = metrics_df['start_timestamp'] - pd.to_timedelta(lower_s, unit='minute')
metrics_df['scale_time_end'] = metrics_df['end_timestamp'] + pd.to_timedelta(upper_e, unit='minute')

# drop the original start and end time_stamps
metrics_df = metrics_df.drop(columns=['start_timestamp', 'end_timestamp', 'relative_position'])

# set the event_key as the index
metrics_df = metrics_df.set_index("event_key")

#### Testing Overlaps

At this point, we have sucessfully transformed those columns by adding and subtracting increments

In [ ]:
# Parses the metrics dataframe rows into a list of tuples 
tuples_events_scale = create_event_tuple_list_from_df(metrics_df)

Now that we have successfully transformed the original dataframe to reflect a broader range, we will run the overlap algorithm on this.

In [ ]:
event_time_overlapping_ids = time_overlaps(tuples_events_scale) # We now have a list of all overlapping event ids

## Proximity Filtering

### Time Slider Setup
The code chunks below will set up a slider that will be used to subset the event data.

Set the time range to be within the minimum and maximum timestamps to setup the time slider

In [ ]:
# Time Lower and Upper Bounds
LOWER = min(metrics_df['scale_time_start'])
UPPER = max(metrics_df['scale_time_end'])

To implement a time slider effect, we will create a new "event" with start and end timestamps. Then, we will allow the user to change where the window is located.

For instance, $\{{id, start, end\}}$ will be designated our time window. To slide across the events, we will add in a increment or decrement function. Then, we will run the overlap_individual function that will be used.


In [ ]:
### Time 
# {'individual', 'position_x', 'position_y', 'scale_time_start', 'scale_time_end'}
# {-1, start, end}: -1 denotes that this is not a bird event, but is user controlled
def time_filter(time_range: tuple, events: tuple) -> bool:
    """
    Returns if user specified time frame overlaps with the current one \n 
    Relies on the sorting invariant
    """
    return time_overlap(time_range, events[0]) and time_overlap(time_range, events[1])

### Proximity Slider Setup

Allow the user to specify the distance that the birds have to be from each other to include in the graph visualization.

$D = \sqrt((Y_2 - Y_1)^2 + (X_2 - X_1)^2)$

In [ ]:
# Specify the distance in meters
EUCLIDEAN_DIST = 30

In [ ]:
# Distance: (X_2, Y_2) and (X_1, Y_1)
def distance_filter(dist_range: int, events: list):
    """
    Returns the Euclidean distance between two events A and B \n
    Does not rely on the sorting invariant
    """
    dist = math.sqrt(events[0]^2 + events[1]^2)
    return dist <= dist_range

## Individual Identification
Using the unique event ids, we will find the corresponding individuals and plot only those events that occurred at the same time.

This function will be re-executed every time the user changes the time & proximity

In [ ]:
def filter_events_individuals(id_list, time, distance):
    # id_list[x]
    '''
    Filters the event list and returns a list of tuples of individuals that overlap. \n
    If time is True, then overlap by that prior \n 
    '''
    individuals = [] # (individual_1, individual_2)
    for id in id_list:
        # changed to metrics_df from subst_df
        event_1 = metrics_df.loc[id[0]] 
        event_2 = metrics_df.loc[id[1]]

        # Fetch the corresponding ids for the overlapping events
        id_event_1 = int(event_1["individual"])  # type: ignore
        id_event_2 = int(event_2["individual"]) # type: ignore

        # Adding the time filter here soon
        t_filter = time_filter(event_1, event_2)

        # Adding the proximity filter here soon
        d_filter = distance_filter(event_1, event_2)

        # Event Pairs must be unique & In the same site & must be in the time and distance ranges specified by the user
        if event_1["array"] != event_2["array"] or id_event_1 == id_event_2 or not t_filter or not d_filter: 
            continue 

        individuals.append(( {"name": id_event_1, "info": event_1}, { "name": id_event_2, "info": event_2} ))
    return individuals
    

## Network Construction

Now we have a list of the individuals with overlapping acoustic records, we have enough information to create visualization of the graphs.

Import the networkX library in order to store the individuals into the graph.

In [ ]:
import networkx as nx

This function will be re-executed when the user changes the time and proximity sliders

In [ ]:
# Now we can iterate over the list of tuples, adding them to the Graph
# ({name, info}, {name, info}) : These two individual birds had an event overlap, each with its own metadata
# This function will be reran with different user configurations for every change
def create_bird_network(individuals_list) -> nx.Graph:
    """
    Using the list of individual overlaps, create a bird network \n
    Computed for each user configuration
    """
    bird_net = nx.Graph()
    for overlap in individuals_list:
        a = overlap[0]["name"]
        b = overlap[1]["name"]
        
        bird_net.add_node(a)
        bird_net.add_node(b)

        if bird_net.has_edge(a, b):
            bird_net[a][b]["weight"] += 1  # increment on repeat overlap
        else:
            bird_net.add_edge(a, b, weight=1)  
    return bird_net

## Network Visualization
Now, we will use Pyvis to visualize the bird interaction network. Since the nodes and edges tend to cluster up and overlap when using matplotlib, Pyvis was chosen due to its repulsion physics that make it more understandable.

In [ ]:
individuals = filter_events_individuals(event_time_overlapping_ids, False, False)

In [ ]:
bird_net = create_bird_network(individuals)

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from pyvis.network import Network

components = list(nx.connected_components(bird_net))
cmap = plt.get_cmap("tab20") # 20 distinct colors

In [ ]:
# Build a site -> color mapping
site_map = {}  # node -> site
for node in bird_net.nodes(): 
    site_map[node] = subst_loaded_events_df[
        subst_loaded_events_df["individual"] == node
    ].iloc[0]["array"]

sites = list(set(site_map.values())) # names of the arrays
indices = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 1]  # 11 distinct indices from tab20
site_colors = {site: cmap(indices[i]) for i, site in enumerate(sites)} # maps each site to color

In [ ]:
net = Network(notebook=True, height="750px", width="100%")

# Add nodes
for i, component in enumerate(components): # Color changes based on components
    for node in component: # Will be the same color
        site = site_map[node]
        color = site_colors[site] # fetch the color for each component
        hex_color = "#{:02x}{:02x}{:02x}".format(
            int(color[0]*255), int(color[1]*255), int(color[2]*255)
        )
        net.add_node(node,
                     label=str(node),
                     color=hex_color,
                     font={"size": 14, "color": "black", "align": "center"})

# Add edges with normalized weights
weights = [data.get("weight", 1) for _, _, data in bird_net.edges(data=True)]
min_w, max_w = min(weights), max(weights)

for src, dest, data in bird_net.edges(data=True):
    w = data.get("weight", 1)
    normalized = 1 + 14 * (w - min_w) / (max_w - min_w)
    net.add_edge(src, dest, value=normalized)

# Toggle the buttons and show the canvas
net.show_buttons(filter_=["physics"])
net.show("bird_net.html")

## Extended Data Set
Prior, we worked with the small Localization data set to gain familiarity. In the following section, we will extend these methods to the larger unannotated data set.

The goal is to create a network from the much larger data set and examine how well the methodology used above generalizes to this. The Snowy HPC at the Kitzes Lab will be the primary medium that will be used to execute the following scripts.

## [TODO] Mathematical Modeling

In order to capture the interactions as precisely as possible, a mathematical model comprising the spatial & temporal aspects of the data will be formulated.

Here are the symbols that will be utilized for this formulations:
- $I$ denotes the interaction between two individual birds
- $I, J$ denote the observed acoustic events of two distinct individual birds
- $\beta_i \in \{\beta_0, \beta_1, \ldots \}$ denote the weights that should be applied
- $S(i, j)$ denotes the Euclidean Distance between i and j in meters, normalized to [0, 1] in which 1 is the closest
- $T(i, j)$ denotes the Temporal Proximity between i and j in minutes, normalized to [0, 1] in which 1 meaning no delay
- $H(i, j)$ denotes the number of historical interactions between i and j, normalized to [0, 1]
- $\theta_{ij} = \beta_0 + \beta_1 \times S(i, j) + \beta_2 \times T(i, j) + \beta_3 \times H(i, j)$ 
- $\sigma = \frac{1}{1 + e^{-\theta_{ij}}}$ denotes the logistic sigmoid function that maps into the range [0, 1]

### Edge Weights

$P(I \mid X, Y) = \sigma [\beta_0 + \beta_1 \times S(X, Y) + \beta_2 \times T(X, Y) + \beta_3 \times H(X, Y)] = \sigma [\theta_{ij}] = z$

In other words, the probability of an interaction occurring given evidence for X and Y is the logistic transformation of the linear combination of the Spatial Temporal Evidence. 

$\beta$ coefficients are free parameters that will be estimated from the data via Bayesian Inference

$P(I \mid X, Y) \propto P(X, Y \mid I) \times P(I)$

The above is the Bayesian formulation for the same phenomena. The probability that an interaction has occurred given evidence from two birds is equivalent to the probability of the evidence given the interaction times the probability of the interaction occurring. The evidence term refers to all the interactions between X & Y, which is left out as it is expensive to calculate in addition to being nearly intractable.

### Future Additions
Identify the angle at which a chirp is coming from the further increase the accuracy of the model. 